<a href="https://colab.research.google.com/github/saintdaniel009-cell/Aizenosa/blob/main/Rigo_Depth_Map_Generator_CPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rigo Depth Map Generator
Makes grayscale depth maps (bright = near, dark = far) for the **Rigo_DepthBlur** Fusion macro.

**First time:** Runtime → Change runtime type → **T4 GPU** → Save.

**Every time:**
1. Drop clips into **Google Drive → `DepthMaps/in`** (folder is created on first run).
2. Run **Cell 1** (setup, ~2 min) then **Cell 2** (process).
3. Collect results from **`DepthMaps/out`** — `<clipname>_depth.mp4`, same length and frame rate as the source. Processed sources move to `DepthMaps/in/done`.

Long clips are handled automatically (resolution is lowered to fit memory — no splitting, no brightness seams).

In [ ]:
#@title 1 · Setup (run once per session)
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/DepthMaps'
for d in ('in', 'in/done', 'out'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

REPO = '/content/Video-Depth-Anything'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/DepthAnything/Video-Depth-Anything', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'easydict', 'decord', 'einops==0.4.1', 'imageio-ffmpeg'], check=True)

# weights (Large; cached in Drive so it only downloads once ever)
os.makedirs(f'{ROOT}/.weights', exist_ok=True)
W = f'{ROOT}/.weights/video_depth_anything_vitl.pth'
if not os.path.exists(W) or os.path.getsize(W) < 1e8:
    subprocess.run(['wget', '-q', '-O', W, 'https://huggingface.co/depth-anything/Video-Depth-Anything-Large/resolve/main/video_depth_anything_vitl.pth'], check=True)
os.makedirs(f'{REPO}/checkpoints', exist_ok=True)
link = f'{REPO}/checkpoints/video_depth_anything_vitl.pth'
if not os.path.exists(link):
    os.symlink(W, link)

# memory-efficient attention patch (fixes CUDA out of memory without xFormers)
p = f'{REPO}/video_depth_anything/dinov2_layers/attention.py'
s = open(p).read()
if 'scaled_dot_product_attention' not in s:
    old = """        q, k, v = qkv[0] * self.scale, qkv[1], qkv[2]
        attn = q @ k.transpose(-2, -1)

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)"""
    new = """        q, k, v = qkv[0], qkv[1], qkv[2]
        x = torch.nn.functional.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C)"""
    assert old in s, 'Repo changed upstream - attention patch needs updating'
    s = s.replace(old, new).replace('from torch import Tensor', 'import torch\nfrom torch import Tensor', 1)
    open(p, 'w').write(s)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU')
print('Ready. Put clips in Drive/DepthMaps/in and run Cell 2.')


Mounted at /content/drive
GPU: Tesla T4
Ready. Put clips in Drive/DepthMaps/in and run Cell 2.


In [ ]:
#@title 2 · Generate depth maps
max_resolution = 1280  #@param {type:"slider", min:480, max:1920, step:32}
#@markdown Lower `max_resolution` only if a clip still crashes. Depth maps don't need full resolution — Resolve scales them up.

import os, sys, glob, shutil, gc, json, subprocess, time
import numpy as np, torch, imageio
sys.path.insert(0, '/content/Video-Depth-Anything'); os.chdir('/content/Video-Depth-Anything')
from video_depth_anything.video_depth import VideoDepthAnything
from utils.dc_utils import read_video_frames

ROOT = '/content/drive/MyDrive/DepthMaps'
EXT = ('.mp4', '.mov', '.m4v', '.mkv', '.mxf', '.avi')

def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-count_packets',
                        '-show_entries', 'stream=width,height,nb_read_packets', '-of', 'json', path],
                       capture_output=True, text=True)
    st = json.loads(r.stdout)['streams'][0]
    return int(st['width']), int(st['height']), int(st['nb_read_packets'])

def pick_res(w, h, n, cap):
    # peak RAM ~12 bytes/pixel/frame (frames + depth list + stacked copy); keep under ~7 GB
    budget = 7e9
    long_side, short = max(w, h), min(w, h)
    for r in range(min(cap, long_side), 383, -32):
        if n * r * (r * short / long_side) * 12 < budget:
            return r
    return 384

def to_h264(src):
    # ffmpeg re-encode to a constant-frame-rate H.264 so decord reads it reliably (iPhone .mov etc.)
    tmp = '/content/_in.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src, '-an', '-c:v', 'libx264', '-crf', '14', '-preset', 'fast', '-pix_fmt', 'yuv420p', tmp], check=True)
    return tmp

model = VideoDepthAnything(encoder='vitl', features=256, out_channels=[256, 512, 1024, 1024])
model.load_state_dict(torch.load('checkpoints/video_depth_anything_vitl.pth', map_location='cpu'), strict=True)
model = model.to('cuda').eval()

clips = sorted(f for f in glob.glob(f'{ROOT}/in/*') if f.lower().endswith(EXT))
if not clips:
    print('No clips found in Drive/DepthMaps/in')
for i, src in enumerate(clips, 1):
    name = os.path.splitext(os.path.basename(src))[0]
    out = f'{ROOT}/out/{name}_depth.mp4'
    t0 = time.time()
    w, h, n = probe(src)
    res = pick_res(w, h, n, max_resolution)
    print(f'[{i}/{len(clips)}] {name}: {w}x{h}, {n} frames -> depth at {res}px')
    tmp = to_h264(src)
    frames, fps = read_video_frames(tmp, -1, -1, res)
    with torch.no_grad():
        depths, fps = model.infer_video_depth(frames, fps, input_size=518, device='cuda', fp32=False)
    del frames; gc.collect(); torch.cuda.empty_cache()
    d_min, d_max = float(depths.min()), float(depths.max())   # one range for the whole clip = no flicker/seams
    wr = imageio.get_writer(out, fps=fps, macro_block_size=1, codec='libx264',
                            ffmpeg_params=['-crf', '12', '-pix_fmt', 'yuv420p'])
    for d in depths:
        wr.append_data(((d - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8))
    wr.close()
    del depths; gc.collect()
    shutil.move(src, f'{ROOT}/in/done/{os.path.basename(src)}')
    print(f'    saved {os.path.basename(out)}  ({time.time()-t0:.0f}s)')
print('Done.')


xFormers not available
xFormers not available
[1/1] IMG_7307: 3840x2160, 351 frames -> depth at 1280px


100%|██████████| 16/16 [02:05<00:00,  7.82s/it]


    saved IMG_7307_depth.mp4  (465s)
Done.


**No GPU?** Set runtime to CPU, run Cell 1, then run **2b** below instead of Cell 2. Test first, then do the full clip.

In [ ]:
#@title 2b · Generate depth maps — CPU mode (no GPU needed)
model_size = 'small'  #@param ['small', 'large']
max_resolution = 640  #@param {type:"slider", min:320, max:1280, step:32}
frame_step = 2  #@param {type:"slider", min:1, max:4, step:1}
test_frames = 24  #@param {type:"integer"}
#@markdown **Test first:** `test_frames = 24` times a short run and saves `<clip>_TEST_depth.mp4` (source stays in `in`). Set to `0` for the full clip.
#@markdown `frame_step = 2` computes every 2nd frame and holds it for 2 frames; output length and fps still match the source.

import os, sys, glob, shutil, gc, json, subprocess, time
import numpy as np, torch, imageio
sys.path.insert(0, '/content/Video-Depth-Anything'); os.chdir('/content/Video-Depth-Anything')
from video_depth_anything.video_depth import VideoDepthAnything
from utils.dc_utils import read_video_frames

ROOT = '/content/drive/MyDrive/DepthMaps'
EXT = ('.mp4', '.mov', '.m4v', '.mkv', '.mxf', '.avi')
CFG = {'small': dict(encoder='vits', features=64, out_channels=[48, 96, 192, 384], file='video_depth_anything_vits.pth',
                     url='https://huggingface.co/depth-anything/Video-Depth-Anything-Small/resolve/main/video_depth_anything_vits.pth'),
       'large': dict(encoder='vitl', features=256, out_channels=[256, 512, 1024, 1024], file='video_depth_anything_vitl.pth',
                     url='https://huggingface.co/depth-anything/Video-Depth-Anything-Large/resolve/main/video_depth_anything_vitl.pth')}[model_size]

W = f"{ROOT}/.weights/{CFG['file']}"
if not os.path.exists(W) or os.path.getsize(W) < 5e7:
    subprocess.run(['wget', '-q', '-O', W, CFG['url']], check=True)
link = f"checkpoints/{CFG['file']}"
if not os.path.exists(link):
    os.symlink(W, link)

def to_h264(src):
    tmp = '/content/_in.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src, '-an', '-c:v', 'libx264', '-crf', '14',
                    '-preset', 'fast', '-pix_fmt', 'yuv420p', tmp], check=True)
    return tmp

model = VideoDepthAnything(encoder=CFG['encoder'], features=CFG['features'], out_channels=CFG['out_channels'])
model.load_state_dict(torch.load(link, map_location='cpu'), strict=True)
model = model.to('cpu').eval()

clips = sorted(f for f in glob.glob(f'{ROOT}/in/*') if f.lower().endswith(EXT))
if not clips:
    print('No clips found in Drive/DepthMaps/in')
for i, src in enumerate(clips, 1):
    name = os.path.splitext(os.path.basename(src))[0]
    is_test = test_frames > 0
    out = f"{ROOT}/out/{name}{'_TEST' if is_test else ''}_depth.mp4"
    t0 = time.time()
    tmp = to_h264(src)
    frames, fps = read_video_frames(tmp, -1, -1, max_resolution)
    n_all = len(frames)
    if is_test:
        frames = frames[:test_frames]
        n_all = len(frames)
    sub = frames[::frame_step]
    print(f'[{i}/{len(clips)}] {name}: {n_all} frames, computing {len(sub)} at {max_resolution}px ({model_size}, CPU)')
    with torch.no_grad():
        depths, _ = model.infer_video_depth(sub, fps, input_size=518, device='cpu', fp32=True)
    del frames, sub; gc.collect()
    dt = time.time() - t0
    if is_test:
        print(f'    test took {dt:.0f}s -> ~{dt/len(depths):.1f}s per computed frame')
    d_min, d_max = float(depths.min()), float(depths.max())
    wr = imageio.get_writer(out, fps=fps, macro_block_size=1, codec='libx264',
                            ffmpeg_params=['-crf', '12', '-pix_fmt', 'yuv420p'])
    written = 0
    for d in depths:
        img = ((d - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8)
        for _ in range(frame_step):
            if written < n_all:
                wr.append_data(img); written += 1
    wr.close()
    del depths; gc.collect()
    if not is_test:
        shutil.move(src, f'{ROOT}/in/done/{os.path.basename(src)}')
    print(f'    saved {os.path.basename(out)}  ({time.time()-t0:.0f}s)')
print('Done.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
